NOT COMPLETE
Prediction approach: trying to predict the next month's values for all the different features.

In [141]:
import pandas as pd
import joblib
import os
import pickle
import skfuzzy as fuzz
import numpy as np
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.vector_ar.var_model import VAR
from sklearn.preprocessing import MinMaxScaler

Method to find cluster of a customer based on the previous data.

In [142]:
def predict_from_array(customer_array, model_file='../models/fuzzy_cmeans_model.pkl'):
    """
    Predicts the cluster for a single record/month data given a preprocessed numpy array.
    
    Parameters:
        customer_array (numpy.ndarray): A 1D numpy array of preprocessed feature values.
        model_file (str): Path to the saved Fuzzy C-Means model.
    
    Returns:
        str: Predicted cluster name.
    """
    if not isinstance(customer_array, np.ndarray) or customer_array.ndim != 1:
        raise ValueError("Input must be a 1D numpy array.")

    # Load the saved model
    fuzzy_cmeans_model = joblib.load(model_file)
    centers = fuzzy_cmeans_model['centers']
    mapping = fuzzy_cmeans_model['mapping']
    cluster_names = fuzzy_cmeans_model['cluster_names']

    # Reshape the input to match the model's expectation
    customer_array = customer_array.reshape(-1, 1)  # Convert to column vector

    # Predict cluster membership
    u_pred, _, _, _, _, _ = fuzz.cluster.cmeans_predict(
        customer_array, centers, fuzzy_cmeans_model['fuzziness_parameter'], 
        error=0.005, maxiter=1000
    )

    # Get hard cluster assignment
    hard_label_pred = np.argmax(u_pred, axis=0)[0]

    # Apply the saved mapping
    renumbered_label_pred = mapping[hard_label_pred]

    # Convert to cluster name
    cluster_prediction = cluster_names.get(renumbered_label_pred, "Unknown")

    return cluster_prediction


In [143]:
# predicts the cluster for a dataframe
def predict_clusters(df_all_features, model_file='../models/fuzzy_cmeans_model.pkl'):
    # features used by segmentation model
    feature_columns = [
    'TotalRevenue', 
    'TotalProfit', 
    'TotalUnitPrice', 
    'OrderCount', 
    'NumDistinctProducts', 
    'BranchCount',  
    'HighProfitOrderRatio', 
    'PopularOrderRatio',
    'NumDistinctProductGroups', 
    'CustomerLifetimeDays', 
    'RepeatPurchaseRatio'
    ]
    # scale features for prediction model
    df = df_all_features[feature_columns]
    standardScaler = StandardScaler()
    normalizedFeatures = standardScaler.fit_transform(df)
    weights = [1,0.57821138, 0.10583493, 0.07332408, 0.07208094, 0.04819159,
    0.01809966, 0.04088511, 0.03574647, 0.01459683, 0.01302901] # values from testing
    weightedFeatures = normalizedFeatures * weights

    # Load the saved model
    fuzzy_cmeans_model = joblib.load(model_file)
    centers = fuzzy_cmeans_model['centers']
    mapping = fuzzy_cmeans_model['mapping']
    cluster_names = fuzzy_cmeans_model['cluster_names']
    u_pred, _, _, _, _, _ = fuzz.cluster.cmeans_predict(
            weightedFeatures.T, centers, fuzzy_cmeans_model['fuzziness_parameter'], 
            error=0.005, maxiter=1000
        )
    # Get hard cluster assignments
    hard_labels_pred = np.argmax(u_pred, axis=0)

    # Apply the saved mapping
    renumbered_labels_pred = mapping[hard_labels_pred]

    # Convert to cluster names
    named_labels_pred = np.vectorize(cluster_names.get)(renumbered_labels_pred)
    # memebership_degree = u_pred
    return pd.Series(named_labels_pred)
    


In [128]:
def add_cluster_prediction(data_file_path, model_file='../models/fuzzy_cmeans_model.pkl'):
    # import data
    df_all_features = pd.read_csv(data_file_path)
    cluster_predictions = predict_clusters(df_all_features, model_file)
    df_all_features['cluster_prediction'] = cluster_predictions
    return df_all_features

Working on the prediction model

In [144]:
file = "../data/cleaned/monthlymetrics/customer_metrics_2022_01_data.csv"
data = pd.read_csv(file)
unique_customers = data["CustomerName"].unique()


In [145]:
input_dir = "../data/cleaned/monthlymetrics"
testing_customer_name = "ADAMS PLC LTD"

# Initialize an empty DataFrame with correct column names
df_for_customer = pd.DataFrame(columns=['CustomerName', 'TotalRevenue', 'TotalProfit', 'TotalUnitPrice',
       'TotalSalesQty', 'OrderCount', 'NumDistinctProducts',
       'NumDistinctProductGroups', 'BranchCount', 'TopProductID',
       'TopProductGroupID', 'ProductDiversityIndex',
       'ProductConcentrationIndex', 'TopProductRevenueShare',
       'TopProductGroupRevenueShare', 'CustomerLifetimeDays', 'AvgOrderValue',
       'RepeatPurchaseRatio', 'HighProfitProductShare', 'PopularProductShare',
       'HighProfitOrderRatio', 'PopularOrderRatio', 'Year', 'Month'])

for file in sorted(os.listdir(input_dir)):  # Sort files to ensure order
    if file.endswith(".csv"):
        file_path = os.path.join(input_dir, file)
        data = pd.read_csv(file_path)

        # Extract year and month from the filename
        data["Year"] = file[17:21]
        data["Month"] = file[22:24]

        # Filter for the specific customer
        customer_data = data[data['CustomerName'] == testing_customer_name]

        # Concatenate the result
        df_for_customer = pd.concat([df_for_customer, customer_data], ignore_index=True)

print(df_for_customer)


     CustomerName  TotalRevenue  TotalProfit  TotalUnitPrice  TotalSalesQty  \
0   ADAMS PLC LTD          0.00         0.00            0.00            0.0   
1   ADAMS PLC LTD          0.00         0.00            0.00            0.0   
2   ADAMS PLC LTD          0.00         0.00            0.00            0.0   
3   ADAMS PLC LTD          0.00         0.00            0.00            0.0   
4   ADAMS PLC LTD          0.00         0.00            0.00            0.0   
5   ADAMS PLC LTD        598.00       113.60          598.00            2.0   
6   ADAMS PLC LTD          0.00         0.00            0.00            0.0   
7   ADAMS PLC LTD          0.00         0.00            0.00            0.0   
8   ADAMS PLC LTD        101.54        37.62           41.37            8.0   
9   ADAMS PLC LTD          0.00         0.00            0.00            0.0   
10  ADAMS PLC LTD          0.00         0.00            0.00            0.0   
11  ADAMS PLC LTD          0.00         0.00        

/var/folders/pf/rh7r7nr92zggr4t5dsl35h2m0000gn/T/ipykernel_60515/722376885.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_for_customer = pd.concat([df_for_customer, customer_data], ignore_index=True)


In [ ]:
input_dir = "../data/cleaned/monthlymetrics/test"
testing_customer_name = "ADAMS PLC LTD"
# Initialize an empty DataFrame with correct column names
df_for_customer_2024 = pd.DataFrame(columns=['CustomerName', 'TotalRevenue', 'TotalProfit', 'TotalUnitPrice',
       'TotalSalesQty', 'OrderCount', 'NumDistinctProducts',
       'NumDistinctProductGroups', 'BranchCount', 'TopProductID',
       'TopProductGroupID', 'ProductDiversityIndex',
       'ProductConcentrationIndex', 'TopProductRevenueShare',
       'TopProductGroupRevenueShare', 'CustomerLifetimeDays', 'AvgOrderValue',
       'RepeatPurchaseRatio', 'HighProfitProductShare', 'PopularProductShare',
       'HighProfitOrderRatio', 'PopularOrderRatio', 'Year', 'Month'])

for file in sorted(os.listdir(input_dir)):  # Sort files to ensure order
    if file.endswith(".csv"):
        file_path = os.path.join(input_dir, file)
        data = pd.read_csv(file_path)

        # Extract year and month from the filename
        data["Year"] = file[17:21]
        data["Month"] = file[22:24]

        # Filter for the specific customer
        customer_data = data[data['CustomerName'] == testing_customer_name]

        # Concatenate the result
        df_for_customer_2024 = pd.concat([df_for_customer, customer_data], ignore_index=True)

In [146]:
feature_columns = [
    'TotalRevenue', 
    'TotalProfit', 
    'TotalUnitPrice', 
    'OrderCount', 
    'NumDistinctProducts', 
    'BranchCount',  
    'HighProfitOrderRatio', 
    'PopularOrderRatio',
    'NumDistinctProductGroups', 
    'CustomerLifetimeDays', 
    'RepeatPurchaseRatio'
    ]

df_for_customer = df_for_customer[feature_columns]


In [147]:
# Fit VAR model (adjust maxlags based on your data)
model = VAR(df_for_customer)
results = model.fit(maxlags=2)  # Experiment with lags (1-4)

In [148]:
# Forecast month 25
forecast = results.forecast(df_for_customer.values[-2:], steps=1)
prediciton = forecast[0]  # Array of 8 feature predictions

In [149]:
print(prediciton)

[38.86333333  8.40111111 35.52055556  0.11111111  0.27777778  0.11111111
  0.          0.          0.22222222  0.          0.0462963 ]


In [150]:
# Create a DataFrame for the prediction record
prediction_record = pd.DataFrame([prediciton], columns=feature_columns)

# Append the prediction record to df_for_customer
df_for_customer_prediction = pd.concat([df_for_customer, prediction_record], ignore_index=True)

print(df_for_customer_prediction.tail(1))

    TotalRevenue  TotalProfit  TotalUnitPrice  OrderCount  \
24     38.863333     8.401111       35.520556    0.111111   

    NumDistinctProducts  BranchCount  HighProfitOrderRatio  PopularOrderRatio  \
24             0.277778     0.111111                   0.0                0.0   

    NumDistinctProductGroups  CustomerLifetimeDays  RepeatPurchaseRatio  
24                  0.222222                   0.0             0.046296  


In [151]:
ground_truth_file = "../data/cleaned/monthlymetrics/test/customer_metrics_2024_01_data.csv"
ground_truth_data = pd.read_csv(ground_truth_file)
# Filter for the specific customer
customer_data = ground_truth_data[ground_truth_data['CustomerName'] == testing_customer_name]
feature_columns = [
    'TotalRevenue', 
    'TotalProfit', 
    'TotalUnitPrice', 
    'OrderCount', 
    'NumDistinctProducts', 
    'BranchCount',  
    'HighProfitOrderRatio', 
    'PopularOrderRatio',
    'NumDistinctProductGroups', 
    'CustomerLifetimeDays', 
    'RepeatPurchaseRatio'
    ]

customer_data = customer_data[feature_columns]


In [152]:
# Append the customer_data DataFrame to df_for_customer
df_for_customer_ground_truth = pd.concat([df_for_customer, customer_data], ignore_index=True)

print(df_for_customer_ground_truth.tail(1))

    TotalRevenue  TotalProfit  TotalUnitPrice  OrderCount  \
24           0.0          0.0             0.0         0.0   

    NumDistinctProducts  BranchCount  HighProfitOrderRatio  PopularOrderRatio  \
24                  0.0          0.0                   0.0                0.0   

    NumDistinctProductGroups  CustomerLifetimeDays  RepeatPurchaseRatio  
24                       0.0                   0.0                  0.0  


In [153]:
print(predict_clusters(df_for_customer_ground_truth).tail(1))
print(predict_clusters(df_for_customer_prediction).tail(1))

24    Rock
dtype: object
24    Bronze
dtype: object


# testing

In [66]:
from tqdm import tqdm  # Optional for progress bar

In [73]:
# Define constants
input_dir = "../data/cleaned/monthlymetrics"
ground_truth_file = "../data/cleaned/monthlymetrics/test/customer_metrics_2024_01_data.csv"
feature_columns = [
    'TotalRevenue', 'TotalProfit', 'TotalUnitPrice', 'OrderCount',
    'NumDistinctProducts', 'BranchCount', 'HighProfitOrderRatio',
    'PopularOrderRatio', 'NumDistinctProductGroups',
    'CustomerLifetimeDays', 'RepeatPurchaseRatio'
]
feature_columns_with_customer_name = [
    'TotalRevenue', 'TotalProfit', 'TotalUnitPrice', 'OrderCount',
    'NumDistinctProducts', 'BranchCount', 'HighProfitOrderRatio',
    'PopularOrderRatio', 'NumDistinctProductGroups',
    'CustomerLifetimeDays', 'RepeatPurchaseRatio', 'CustomerName'
]

# Load ground truth data
ground_truth_data = pd.read_csv(ground_truth_file)

# Initialize dictionary to store customer data
customer_data_dict = {}

# First pass: collect all data into dictionary
for file in tqdm(sorted(os.listdir(input_dir)), desc="Loading files"):
    if file.endswith(".csv") and "test" not in file:
        file_path = os.path.join(input_dir, file)
        data = pd.read_csv(file_path)
        data = data[feature_columns_with_customer_name]
        data["Year"] = file[17:21]
        data["Month"] = file[22:24]
        
        # Split data by customer and add to dictionary
        for customer_name, customer_data in data.groupby('CustomerName'):
            if customer_name in customer_data_dict:
                customer_data_dict[customer_name] = pd.concat(
                    [customer_data_dict[customer_name], customer_data],
                    ignore_index=True
                )
            else:
                customer_data_dict[customer_name] = customer_data.copy()

# Process predictions
results = []
for customer_name in tqdm(customer_data_dict.keys(), desc="Processing customers"):
    df_customer = customer_data_dict[customer_name]
    
    if len(df_customer) < 3:
        continue  # Not enough data for VAR

    try:
        df_features = df_customer[feature_columns]
        
        # Identify varying columns (non-constant), handling NaNs
        varying_columns = []
        for col in feature_columns:
            if df_features[col].isna().all():
                continue  # Skip if all NaN
            if df_features[col].nunique(dropna=True) > 1:
                varying_columns.append(col)
        
        # Debugging: Print problematic data
        if not varying_columns:
            print(f"Skipped {customer_name} due to no varying features. Data:\n{df_features}")
            continue
        
        # If all columns are constant or too few vary, use last values
        if len(varying_columns) < 2:  # VAR needs at least 2 series
            predicted_features = df_features.iloc[-1][feature_columns].to_numpy()
        else:
            # Train VAR on varying columns only
            df_features_varying = df_features[varying_columns].dropna()  # Drop rows with NaN
            if len(df_features_varying) < 3:
                print(f"Skipped {customer_name} due to insufficient data after NaN removal")
                continue
                
            var_model = VAR(df_features_varying)
            var_result = var_model.fit(maxlags=2)
            forecast = var_result.forecast(df_features_varying.values[-2:], steps=1)
            predicted_features_varying = forecast[0]

            # Combine forecast with last known values for constant columns
            predicted_features = np.zeros(len(feature_columns))
            last_values = df_features.iloc[-1]  # Last row for constant values
            for i, col in enumerate(feature_columns):
                if col in varying_columns:
                    predicted_features[i] = predicted_features_varying[varying_columns.index(col)]
                else:
                    predicted_features[i] = last_values[col] if not pd.isna(last_values[col]) else 0

        # Predict cluster from full feature array
        predicted_cluster = predict_from_array(predicted_features)

        # Get actual features and cluster from ground truth
        actual_data = ground_truth_data[ground_truth_data['CustomerName'] == customer_name]
        if not actual_data.empty:
            actual_features = actual_data[feature_columns].to_numpy()[0]
            actual_cluster = predict_from_array(actual_features)

            # Store comparison result
            results.append({
                "CustomerName": customer_name,
                "PredictedCluster": predicted_cluster,
                "ActualCluster": actual_cluster,
                "Match": predicted_cluster == actual_cluster
            })

    except Exception as e:
        print(f"Skipped {customer_name} due to error: {e}")
        # Debugging: Print the varying columns and data
        print(f"Varying columns: {varying_columns}")
        print(f"Data for VAR:\n{df_features[varying_columns]}")

# Convert results to DataFrame
comparison_df = pd.DataFrame(results)
print(comparison_df.head())

# Print summary accuracy
accuracy = comparison_df["Match"].mean()
print(f"Prediction Accuracy: {accuracy:.2%}")

Processing customers:   9%|▉         | 96/1039 [00:00<00:01, 483.60it/s]

Skipped ALLEN, JOHNSON AND SAUNDERS GROUP due to no varying features. Data:
    TotalRevenue  TotalProfit  TotalUnitPrice  OrderCount  \
0            0.0          0.0             0.0         0.0   
1            0.0          0.0             0.0         0.0   
2            0.0          0.0             0.0         0.0   
3            0.0          0.0             0.0         0.0   
4            0.0          0.0             0.0         0.0   
5            0.0          0.0             0.0         0.0   
6            0.0          0.0             0.0         0.0   
7            0.0          0.0             0.0         0.0   
8            0.0          0.0             0.0         0.0   
9            0.0          0.0             0.0         0.0   
10           0.0          0.0             0.0         0.0   
11           0.0          0.0             0.0         0.0   
12           0.0          0.0             0.0         0.0   
13           0.0          0.0             0.0         0.0   
14       

Processing customers:  25%|██▍       | 255/1039 [00:00<00:01, 516.12it/s]

Skipped CARR LLC GROUP due to error: x contains one or more constant columns. Column(s) 16 are constant. Adding a constant with trend='c' is not allowed.
Varying columns: ['TotalRevenue', 'TotalProfit', 'TotalUnitPrice', 'OrderCount', 'NumDistinctProducts', 'BranchCount', 'HighProfitOrderRatio', 'PopularOrderRatio', 'NumDistinctProductGroups', 'CustomerLifetimeDays', 'RepeatPurchaseRatio']
Data for VAR:
    TotalRevenue  TotalProfit  TotalUnitPrice  OrderCount  \
0        2986.60       800.37      822.820433         8.0   
1        5494.29      1798.19      841.093783         8.0   
2        2201.17       614.24     1040.071033         7.0   
3        3742.83      1197.97     1011.724500         7.0   
4        7543.88      2575.45      766.858933        10.0   
5        2894.94       926.46     1601.054733        11.0   
6        1947.53       585.79      821.218667         4.0   
7        3127.81       989.16     1168.371900         8.0   
8        4885.17      1749.96      897.39066

Processing customers:  30%|██▉       | 307/1039 [00:00<00:02, 354.51it/s]

...

Skipped POOLE-MILES PLC due to error: x contains one or more constant columns. Column(s) 15 are constant. Adding a constant with trend='c' is not allowed.
Varying columns: ['TotalRevenue', 'TotalProfit', 'TotalUnitPrice', 'OrderCount', 'NumDistinctProducts', 'BranchCount', 'HighProfitOrderRatio', 'NumDistinctProductGroups', 'CustomerLifetimeDays', 'RepeatPurchaseRatio']
Data for VAR:
    TotalRevenue  TotalProfit  TotalUnitPrice  OrderCount  \
0        1618.81       574.79      476.123233         5.0   
1         527.59       185.95      283.556650         4.0   
2        1758.83       659.69      278.821500         6.0   
3         958.28       373.55      212.816100         5.0   
4        2083.11       772.40      510.767283         8.0   
5        1560.21       593.05      451.352417         5.0   
6        2209.48      1003.55      914.699772         6.0   
7        1646.75       694.73      808.354294         4.0   
8        1023.18       391.37      385.830467         5.0   
9  

Processing customers:  98%|█████████▊| 1022/1039 [00:02<00:00, 471.08it/s]

Skipped TURNER-METCALFE LTD due to no varying features. Data:
    TotalRevenue  TotalProfit  TotalUnitPrice  OrderCount  \
0            0.0          0.0             0.0         0.0   
1            0.0          0.0             0.0         0.0   
2            0.0          0.0             0.0         0.0   
3            0.0          0.0             0.0         0.0   
4            0.0          0.0             0.0         0.0   
5            0.0          0.0             0.0         0.0   
6            0.0          0.0             0.0         0.0   
7            0.0          0.0             0.0         0.0   
8            0.0          0.0             0.0         0.0   
9            0.0          0.0             0.0         0.0   
10           0.0          0.0             0.0         0.0   
11           0.0          0.0             0.0         0.0   
12           0.0          0.0             0.0         0.0   
13           0.0          0.0             0.0         0.0   
14           0.0       

Processing customers: 100%|██████████| 1039/1039 [00:02<00:00, 443.14it/s]

                      CustomerName PredictedCluster ActualCluster  Match
0               ADAMS PLC AND SONS             Rock          Rock   True
1                    ADAMS PLC LTD         Platinum          Rock  False
2  AHMED, MURPHY AND STEVENSON LLC             Rock          Rock   True
3               AHMED-BAILEY GROUP         Platinum          Rock  False
4                AKHTAR-AKHTAR LTD         Platinum          Rock  False
Prediction Accuracy: 40.02%


In [76]:
print(comparison_df.shape)

(982, 4)


In [77]:
unique_actual_clusters = set(result['ActualCluster'] for result in results)
print(unique_actual_clusters)

{'Gold', 'Platinum', 'Rock'}


In [78]:
unique_predicted_clusters = set(result['PredictedCluster'] for result in results)
print(unique_predicted_clusters)

{'Gold', 'Platinum', 'Bronze', 'Rock', 'Silver'}


In [42]:
# Create a DataFrame for the prediction record
prediction_record = pd.DataFrame([prediciton], columns=feature_columns)

# Append the prediction record to df_for_customer
df_for_customer = pd.concat([df_for_customer, prediction_record], ignore_index=True)

print(df_for_customer.tail())

    TotalRevenue   TotalProfit  TotalUnitPrice  OrderCount  \
21  0.000000e+00  0.000000e+00    0.000000e+00         0.0   
22  0.000000e+00  0.000000e+00    0.000000e+00         0.0   
23  0.000000e+00  0.000000e+00    0.000000e+00         0.0   
24  1.998401e-15  1.110223e-15    4.440892e-16         0.0   
25  1.998401e-15  1.110223e-15    4.440892e-16         0.0   

    NumDistinctProducts   BranchCount  HighProfitOrderRatio  \
21         0.000000e+00  0.000000e+00          0.000000e+00   
22         0.000000e+00  0.000000e+00          0.000000e+00   
23         0.000000e+00  0.000000e+00          0.000000e+00   
24         6.071532e-18  8.131516e-20          2.385245e-18   
25         6.071532e-18  8.131516e-20          2.385245e-18   

    PopularOrderRatio  NumDistinctProductGroups  CustomerLifetimeDays  \
21       0.000000e+00              0.000000e+00                   0.0   
22       0.000000e+00              0.000000e+00                   0.0   
23       0.000000e+00        

In [43]:
standardScaler = StandardScaler()
normalizedFeatures = standardScaler.fit_transform(df_for_customer)
weights = [1,0.57821138, 0.10583493, 0.07332408, 0.07208094, 0.04819159,
0.01809966, 0.04088511, 0.03574647, 0.01459683, 0.01302901] # values from testing
weightedFeatures = normalizedFeatures * weights
# Load the saved model
fuzzy_cmeans_model = joblib.load('../models/fuzzy_cmeans_model.pkl')
centers = fuzzy_cmeans_model['centers']
mapping = fuzzy_cmeans_model['mapping']
cluster_names = fuzzy_cmeans_model['cluster_names']
u_pred, _, _, _, _, _ = fuzz.cluster.cmeans_predict(
        weightedFeatures.T, centers, fuzzy_cmeans_model['fuzziness_parameter'], 
        error=0.005, maxiter=1000
    )
# Get hard cluster assignments
hard_labels_pred = np.argmax(u_pred, axis=0)

# Apply the saved mapping
renumbered_labels_pred = mapping[hard_labels_pred]

# Convert to cluster names
named_labels_pred = np.vectorize(cluster_names.get)(renumbered_labels_pred)
# memebership_degree = u_pred
df_for_customer['cluster_prediction'] = pd.Series(named_labels_pred)

In [44]:
df_for_customer['cluster_prediction'].tail()

21    Rock
22    Rock
23    Rock
24    Rock
25    Rock
Name: cluster_prediction, dtype: object